In [9]:
import pandas as pd

df = pd.read_csv('../../files/results/data.csv').drop(columns=['Unnamed: 0', 'ano_primeiro_registro', 'ano_ingresso_final',
       'periodo_primeiro', 'periodo_ingresso_final', 'percentil_media',
       'reprovado', 'is_trancado', 'target_percntil_media', 'target_reprov',
       'target_trancamento', 'risco', 'target_risco'])

In [14]:
df.columns

Index(['id_discente', 'ano', 'periodo', 'id_disciplina', 'situacao', 'sexo',
       'estado_civil', 'raca_declarada', 'discente_nivel', 'id_curso',
       'id_curriculo', 'id_estrutura_curricular', 'ano_ingresso',
       'periodo_ingresso', 'status_discente', 'forma_ingresso',
       'quantidade_membros_familia', 'ch_integralizada', 'ch_pendente',
       'media_geral', 'ano_nascimento', 'faixa_renda_familiar',
       'uf_titulo_eleitor_pb', 'uf_naturalidade_pb', 'pais_origem_br',
       'id_detalhe', 'nome', 'ch_aula', 'ch_laboratorio', 'ch_total',
       'cr_aula', 'cr_laboratorio', 'cr_estagio', 'ch_ead',
       'sigla_departamento', 'nivel_componente_curricular', 'sigla_academica',
       'nome_departamento', 'sigla_centro', 'nome_centro',
       'qtd_max_matriculas', 'codigo_componente_curricular',
       'nome_componete_curricular', 'descricao_tipo_componente_curricular',
       'excluir_avaliacao_institucional', 'ativo', 'curso_nome',
       'curso_unidade_nome', 'campus', 'curso

primeira target: média por percentil

In [11]:
# target 1, de maneira simples usaremos a média do aluno, o ruim é que não captura tendência

# analisando por tamanho qual a melhor composição para agregar pela média
print(df.groupby(['curso_nome']).size().describe())
print(df.groupby(['curso_nome', 'periodo']).size().describe())
print(df.groupby(['curso_nome', 'ano', 'periodo']).size().describe())

count      21.000000
mean      825.809524
std       763.068190
min        14.000000
25%       234.000000
50%       585.000000
75%      1449.000000
max      2583.000000
dtype: float64
count      39.000000
mean      444.666667
std       373.445715
min        14.000000
25%       128.000000
50%       300.000000
75%       739.500000
max      1292.000000
dtype: float64
count    281.000000
mean      61.715302
std       40.410007
min        1.000000
25%       29.000000
50%       57.000000
75%       88.000000
max      202.000000
dtype: float64


Justificativa:

A escolha do nível de agregação da variável target foi baseada no trade-off entre viés e variância.

O agrupamento por curso apresenta baixa variância da média devido ao alto número de observações por grupo (média ≈ 826), porém introduz alto viés ao ignorar variações temporais relevantes para o desempenho discente.

O agrupamento por curso + ano + período, apesar de capturar dinâmicas temporais importantes, apresenta alta variância (média ≈ 62 observações por grupo, com casos mínimos de 1), tornando as estimativas instáveis e suscetíveis a ruído.

O agrupamento por curso + período representa um compromisso adequado, mantendo tamanho amostral suficiente (média ≈ 445) para garantir estabilidade estatística, enquanto incorpora parcialmente a progressão acadêmica, relevante para o problema de predição de risco acadêmico.

Dessa forma, essa agregação minimiza o erro total ao equilibrar viés e variância.

In [20]:
# criando variavel do primeiro quartil da média
q1 = df.groupby(['curso_nome', 'periodo'])['media_geral'].transform(lambda x: x.quantile(0.25))

# criando target
df['target_media_percentil'] = (df['media_geral'] <= q1).astype(int)
df['target_media_percentil'].value_counts()

target_media_percentil
0    44211
1     4236
Name: count, dtype: int64

Você quer:

> identificar alunos no **primeiro quartil da distribuição de médias** dentro de cada grupo *(curso, período)*

Formalmente:

[
Q1_{c,p} = \text{percentil 25 da média no grupo } (curso=c, periodo=p)
]

E a target:

[
y_i =
\begin{cases}
1, & \text{se } media_i \le Q1_{c,p} \
0, & \text{caso contrário}
\end{cases}
]

Por que NÃO usar média absoluta (argumento importante)

❌ Threshold fixo (ex: média < 5)

Problemas:

* ignora heterogeneidade entre cursos
* pode enviesar áreas mais difíceis

❌ Z-score global

Problemas:

* mistura distribuições incompatíveis
* assume homogeneidade que não existe

A variável target foi definida com base no primeiro quartil da distribuição da média geral dos discentes dentro de cada grupo de curso e período. Essa abordagem permite identificar alunos com desempenho relativamente baixo dentro de seu contexto acadêmico específico, evitando comparações diretas entre cursos com distribuições de notas distintas.

Além disso, o uso de quartis garante robustez a outliers e produz uma variável balanceada, facilitando o treinamento de modelos preditivos. A segmentação por curso e período também permite capturar variações estruturais e temporais no desempenho acadêmico, tornando a abordagem mais sensível a mudanças no contexto educacional.